## INTRO & SETTINGS

This tutorial mirrors `posterior_posterior_2Dgaussian_mean.ipynb` but replaces the Gaussian likelihood with a two-component **Gaussian mixture** likelihood:

$$\mathbf{X}|\boldsymbol{\theta} \sim 0.5\,\mathcal{N}(\boldsymbol{\theta},\,\mathbb{I}) + 0.5\,\mathcal{N}(\boldsymbol{\theta},\,0.01\,\mathbb{I})$$

The prior is an isotropic Gaussian:

$$\boldsymbol{\theta} \sim \mathcal{N}(\mathbf{0},\,\mathbb{I})$$

We observe $n=1$ sample per true $\boldsymbol{\theta}$ and infer a 2D location parameter using `FMPE` as the posterior estimator.

In [ ]:
# SETTINGS

MIXTURE_WEIGHTS = [0.5, 0.5]   # two-component mixture
MIXTURE_SCALES  = [10.0, 0.1]   # component standard deviations

PRIOR_LOC = 0.0
PRIOR_COV = 2.0

PARAM_DIM  = 2
DATA_DIM   = 2
BATCH_SIZE = 1   # one observed sample per true parameter
PARAM_SPACE_BOUNDS = {'low': -10.0, 'high': 10.0}

CONFIDENCE_LEVEL = 0.90, 0.80, 0.70, 0.60, 0.50, 0.40, 0.30, 0.20, 0.10

## SIMULATE

In [ ]:
import torch
from lf2i.simulator.gmm import GaussianMixtureLocation

In [ ]:
simulator = GaussianMixtureLocation(
    poi_space_bounds=PARAM_SPACE_BOUNDS,
    poi_grid_size=10_000,
    poi_dim=PARAM_DIM,
    data_dim=DATA_DIM,
    batch_size=BATCH_SIZE,
    mixture_weights=torch.tensor(MIXTURE_WEIGHTS),
    mixture_scales=torch.tensor(MIXTURE_SCALES),
    prior_kwargs={'loc': PRIOR_LOC, 'cov': PRIOR_COV},
)

#### Observations

Two "observed" samples: one near the prior centre ($\boldsymbol{\theta}^\star = [0, 0]$) and one in the prior tail ($\boldsymbol{\theta}^\star = [3.0, -3.0]$).

In [ ]:
true_param_consistent    = torch.tensor([0.0,  0.0])
true_param_notconsistent = torch.tensor([-8.5, 8.5])

observed_x_consistent    = simulator.likelihood(true_param_consistent).sample((BATCH_SIZE,))    # (batch_size, data_dim)
observed_x_notconsistent = simulator.likelihood(true_param_notconsistent).sample((BATCH_SIZE,))

print('observed_x_consistent   :', observed_x_consistent)
print('observed_x_notconsistent:', observed_x_notconsistent)

## CONFIDENCE SET via POSTERIOR ESTIMATOR

`Posterior` wraps `FMPE` and guarantees the desired marginal coverage regardless of the prior, the true parameter value, or observed-sample size.

In [ ]:
from lf2i.inference import LF2I
from lf2i.test_statistics import Posterior
from lf2i.utils.other_methods import hpd_region
from lf2i.plot.parameter_regions import plot_parameter_regions

from sbi.inference import FMPE

In [ ]:
lf2i = LF2I(
    test_statistic=Posterior(
        estimator=FMPE(prior=simulator.prior),
        poi_dim=PARAM_DIM,
        n_jobs=1,
    )
)

In [ ]:
confidence_region, point_estimate = lf2i.inference(
    x=torch.vstack((observed_x_consistent, observed_x_notconsistent)),
    evaluation_grid=simulator.poi_grid,
    confidence_level=CONFIDENCE_LEVEL,
    calibration_method='p-values',
    calibration_model='parametric-nn',
    simulator=simulator,
    b=20_000,
    b_prime=10_000,
    recalibrate_p_values=True,
    return_point_estimate=True
)

### Observation consistent with the prior

LF2I Confidence Region

In [ ]:
plot_parameter_regions(
    confidence_region[0],
    param_dim=PARAM_DIM,
    true_parameter=true_param_consistent,
    param_names=['theta0', 'theta1'],
    parameter_space_bounds={'theta0': simulator.poi_space_bounds, 'theta1': simulator.poi_space_bounds},
    alpha_shape=True,
    alpha=2,
    scatter=False,
    figsize=(7.5, 7.5),
    colors=['mediumseagreen'],
    region_names=['LF2I confidence set'],
)

Posterior Credible Region

In [ ]:
plot_parameter_regions(
    hpd_region(
        posterior=lf2i.test_statistic.estimator,
        param_grid=simulator.poi_grid,
        x=observed_x_consistent,
        credible_level=CONFIDENCE_LEVEL,
    )[1],
    param_dim=PARAM_DIM,
    true_parameter=true_param_consistent,
    param_names=['theta0', 'theta1'],
    parameter_space_bounds={'theta0': simulator.poi_space_bounds, 'theta1': simulator.poi_space_bounds},
    alpha_shape=True,
    alpha=2,
    scatter=False,
    figsize=(7.5, 7.5),
    colors=['blue'],
    region_names=['Posterior credible set'],
)

### Observation *not* consistent with the prior

LF2I Confidence Region

In [ ]:
plot_parameter_regions(
    confidence_region[1],
    param_dim=PARAM_DIM,
    true_parameter=true_param_notconsistent,
    param_names=['theta0', 'theta1'],
    parameter_space_bounds={'theta0': simulator.poi_space_bounds, 'theta1': simulator.poi_space_bounds},
    alpha_shape=True,
    alpha=2,
    scatter=False,
    figsize=(7.5, 7.5),
    colors=['mediumseagreen'],
    region_names=['LF2I confidence set'],
)

Posterior Credible Region

In [ ]:
plot_parameter_regions(
    hpd_region(
        posterior=lf2i.test_statistic.estimator,
        param_grid=simulator.poi_grid,
        x=observed_x_notconsistent,
        credible_level=CONFIDENCE_LEVEL,
    )[1],
    param_dim=PARAM_DIM,
    true_parameter=true_param_notconsistent,
    param_names=['theta0', 'theta1'],
    parameter_space_bounds={'theta0': simulator.poi_space_bounds, 'theta1': simulator.poi_space_bounds},
    alpha_shape=True,
    alpha=2,
    scatter=False,
    figsize=(7.5, 7.5),
    colors=['blue'],
    region_names=['Posterior credible set'],
)

## CONFIDENCE DISTRIBUTIONS

In [ ]:
# TODO: Plot 1D confidence distributions

## COVERAGE DIAGNOSTICS

In [ ]:
from lf2i.plot.coverage_diagnostics import coverage_probability_plot

#### Posterior Credible Regions

In [ ]:
diagnostic_estimator, parameters, mean_proba, upper_proba, lower_proba, sizes = lf2i.diagnostics(
    region_type='posterior',
    simulator=simulator,
    b_double_prime=10_000,
    evaluation_grid=simulator.poi_grid.reshape(-1, PARAM_DIM),
    confidence_level=CONFIDENCE_LEVEL,
    posterior_estimator=lf2i.test_statistic.estimator,
    n_jobs=-2,
)

In [ ]:
coverage_probability_plot(
    parameters=parameters,
    coverage_probability=mean_proba,
    upper_proba=None,
    lower_proba=None,
    confidence_level=CONFIDENCE_LEVEL,
    param_dim=PARAM_DIM,
    figsize=(12, 12),
)

#### LF2I Confidence Regions

In [ ]:
diagnostic_estimator, parameters, mean_proba, upper_proba, lower_proba = lf2i.diagnostics(
    region_type='lf2i',
    confidence_level=CONFIDENCE_LEVEL,
    calibration_method='critical-values',
    simulator=simulator,
    b_double_prime=10_000,
)

In [ ]:
coverage_probability_plot(
    parameters=parameters,
    coverage_probability=mean_proba,
    upper_proba=None,
    lower_proba=None,
    confidence_level=CONFIDENCE_LEVEL,
    param_dim=PARAM_DIM,
    figsize=(12, 10),
)

## CALIBRATION DIAGNOSTICS

In [ ]:
# TODO: Plot calibration diagnostics